# Travaux Pratiques : Réseaux de Neurones Récurrents (RNN & LSTM)

**Module** : Apprentissage Séquentiel  
**Niveau** : Master Recherche IA  
**Objectifs** :
1. Implémenter le passage avant d'un RNN standard.
2. Analyser la dynamique des gradients (BPTT) et les phénomènes de Vanishing/Exploding Gradient.
3. Implémenter une cellule LSTM complète.
4. Comprendre le rôle des portes et du tapis roulant (cell state) dans le LSTM.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuration des graphiques
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('bmh')

## 1. Fondements des Réseaux Récurrents

### Exercice 1 — Implémentation du Forward Pass

L'équation de mise à jour de l'état caché est :
$$h_t = \tanh(W_{hx} x_t + W_{hh} h_{t-1} + b_h)$$
La sortie est donnée par :
$$\hat{y}_t = W_{yh} h_t + b_y$$

In [ ]:
np.random.seed(42)
W_hx = np.random.randn(4, 3) * 0.1  # (hidden=4, input=3)
W_hh = np.random.randn(4, 4) * 0.1  # (hidden=4, hidden=4)
b_h = np.zeros(4)
T = 5

def rnn_forward(X, h_init):
    """Passage avant sur une séquence X de shape (T, input_size)."""
    hidden_states = []
    h = h_init
    for t in range(T):
        # TODO : implémentation de l'équation standard
        h = np.tanh(np.dot(W_hx, X[t]) + np.dot(W_hh, h) + b_h)
        hidden_states.append(h.copy())
    return hidden_states

# Test du code
X = np.random.randn(T, 3)
h0 = np.zeros(4)
states = rnn_forward(X, h0)

print(f"Norme de h_T : {np.linalg.norm(states[-1]):.4f}")
print(f"Valeurs h_T : {states[-1].round(4)}")

### Questions :
**b) Quelle est la sortie attendue pour la norme de $h_T$ avec seed=42 ?**  
La sortie attendue, telle qu'indiquée dans le sujet, est **0.3847**.

**c) Que se passe-t-il si $h_0 = \text{np.ones}(4)$ ?**  
Si l'état initial est un vecteur de uns, les premières activations seront plus proches de la saturation de la fonction $\tanh$. Cela peut influencer la dynamique initiale, mais comme les poids sont petits ($*0.1$), le réseau tendra rapidement vers un régime stable dicté par les entrées $X$ et la structure des poids.

## 2. Simulation — Dynamique du RNN

### Exercice 2 — Simulation de la dynamique des états cachés

In [ ]:
np.random.seed(0)

def simulate_rnn(w_norm, T=30, noise_std=0.5):
    h, states = 0.5, []
    for t in range(T):
        x = noise_std * np.random.randn() + np.sin(t * 0.3)
        h = np.tanh(w_norm * h + 0.3 * x)
        states.append(h)
    return np.array(states)

regimes = {
    "Exploding (||W||=1.3)": (1.3, "red"),
    "Stable (||W||=0.8)": (0.8, "limegreen"),
    "Vanishing (||W||=0.3)": (0.3, "orange"),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Dynamique des états cachés selon ||W_hh||", fontsize=15)

for ax, (label, (w, color)) in zip(axes, regimes.items()):
    states = simulate_rnn(w_norm=w)
    ax.plot(states, color=color, linewidth=2)
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Pas t")
    ax.set_ylabel("h_t")

plt.tight_layout()
plt.show()

### Questions :
**a) Description qualitative des courbes :**
- **Exploding (1.3)** : Les états présentent des oscillations de grande amplitude et saturent rapidement vers les bornes $\pm 1$. Le système est instable.
- **Stable (0.8)** : Les états suivent la dynamique de l'entrée avec un certain filtrage. Ils restent informatifs sans saturer ni disparaître.
- **Vanishing (0.3)** : L'effet du passé s'estompe très vite. Les états tendent rapidement vers 0 si l'entrée est faible, perdant la mémoire à long terme.

**b) Modifiez $T=100$ dans le régime explosif :**
Avec $T=100$, la valeur de `states[-1]` oscillera entre les plateaux de saturation. La norme restera proche de 1 (valeur absolue), mais la dynamique sera chaotique.

**c) Quelle technique simple pallie l'explosion ?**  
Le **Gradient Clipping** est la technique privilégiée. En Python : `grad = np.clip(grad, -threshold, threshold)`.

## 3. Vanishing & Exploding Gradient — BPTT

L'approximation spectrale nous donne :
$$\left\|\frac{\partial \mathcal{L}_T}{\partial h_k}\right\| \approx \left\|\frac{\partial \mathcal{L}_T}{\partial h_T}\right\| \cdot |\lambda_{max}(W_{hh})|^{T-k}$$

### Exercice 3 — Calcul de la BPTT

In [ ]:
def bptt_gradient_norm(lambda_max, T):
    """Formule spectrale simplifiée : ||dL/dh_k|| ~ |lambda_max|^(T-k)"""
    grad_norms = []
    for t in range(T):
        # t représente l'indice k dans la formule (T-k)
        norm = 1.0 * (lambda_max ** (T - t - 1))
        grad_norms.append(norm)
    return np.array(grad_norms)

T_len = 20
plt.figure(figsize=(10, 6))
for lam, label, color in [(0.8, "Vanishing", "orange"), (1.0, "Stable", "limegreen"), (1.2, "Exploding", "red")]:
    norms = bptt_gradient_norm(lam, T_len)
    plt.semilogy(norms, label=f"{label} (λ={lam})", color=color)

plt.title("Évolution de la norme du gradient (Backpropagation Through Time)")
plt.xlabel("Pas de temps k")
plt.ylabel("||∂L/∂h_k||")
plt.legend()
plt.grid(True, which="both", ls="--")
plt.show()

### Questions :
**b) Pour $\lambda = 0.95$ et $T = 100$, calculez analytiquement la norme :**
$$\text{Norme} = 0.95^{99} \approx 0.0062$$
Le gradient est divisé par plus de 150 par rapport à sa valeur initiale, illustrant la difficulté de propager l'erreur sur 100 pas même avec un $\lambda$ proche de 1.

**c) Implémentez le gradient clipping :**
```python
grad = np.clip(grad, -5, 5)
```

## 4. La Cellule LSTM (Long Short-Term Memory)

Le LSTM introduit un "tapis roulant" $c_t$ (cell state) protégé par des portes.

### Exercice 4 — Implémentation de la cellule LSTM

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def lstm_cell(x_t, h_prev, c_prev, W, b):
    inp = np.concatenate([x_t, h_prev])
    z = W @ inp + b
    m = h_prev.shape[0]
    
    # 4 Portes
    f_t = sigmoid(z[:m])          # Oubli
    i_t = sigmoid(z[m:2*m])       # Entrée
    c_tilde = np.tanh(z[2*m:3*m]) # Candidat
    o_t = sigmoid(z[3*m:])        # Sortie
    
    # Mises à jour
    c_t = f_t * c_prev + i_t * c_tilde
    h_t = o_t * np.tanh(c_t)
    
    return h_t, c_t

# Test numérique
d, m = 3, 4
W = np.random.randn(4*m, d+m) * 0.1
b = np.zeros(4*m)
h_prev, c_prev = np.zeros(m), np.zeros(m)
x_t = np.ones(d)

h_t, c_t = lstm_cell(x_t, h_prev, c_prev, W, b)
print(f"h_t : {h_t.round(4)}")
print(f"c_t : {c_t.round(4)}")

### Questions :
**a) Pourquoi les portes utilisent-elles $\sigma$ et non $\tanh$ ?**  
La fonction sigmoïde a un ensemble de départ $(0, 1)$, ce qui permet d'agir comme un interrupteur ou un pourcentage (ex: laisser passer 80% de l'information). $\tanh$ varie entre -1 et 1, ce qui permettrait d'inverser le sens de l'information, ce qui n'est pas souhaitable pour une porte de contrôle.

**b) Si $f_t = 0$ partout, à quelle architecture se réduit le LSTM ?**  
Le LSTM se réduit à une architecture sans mémoire à long terme où l'état de cellule est réinitialisé à chaque pas avec seulement les nouvelles informations filtrées par $i_t$.

**c) Calculez le nombre de paramètres pour $d=128, m=256$ :**
- Un bloc de poids pour les 4 portes : $4 \times (m \times (d + m) + m)$
- $4 \times (256 \times (128 + 256) + 256) = 4 \times (256 \times 384 + 256) = 4 \times (98304 + 256) = 394,240$ paramètres.
- En comparaison, un RNN standard possède $m \times (d + m) + m = 98,560$ paramètres.
- Le LSTM possède donc exactement **4 fois plus de paramètres**.

## 5. Quiz de Validation

1. **Pourquoi le RNN classique souffre-t-il du Vanishing Gradient ?**  
   **Réponse : B.** Le gradient est multiplié par $W_{hh}$ à chaque pas ; si $|\lambda_{max}| < 1$, sa norme décroît exponentiellement avec la profondeur temporelle.

2. **Quelle équation justifie que le LSTM résiste au Vanishing Gradient ?**  
   **Réponse : B.** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$. Cette structure additive (appelée *Constant Error Carousel*) permet au gradient de circuler de manière linéaire dans l'état de cellule sans être systématiquement écrasé par des multiplications matricielles répétées.

3. **Si $f_t = 0$ pour tout $t$, à quelle architecture le LSTM se réduit-il ?**  
   **Réponse : C.** Un réseau qui réinitialise $c_t$ à chaque pas.

4. **Quel est le principal avantage d'un GRU par rapport à un LSTM ?**  
   **Réponse : C.** Le GRU a environ 25% de paramètres en moins (fusion des portes d'oubli et d'entrée), ce qui le rend plus rapide à entraîner pour des performances souvent comparables.